This page walks through the everyday calls: find your Lambda layers, list their versions, and fetch one version. It is for anyone who already uses boto3 and wants objects instead of raw dictionaries.

## Setup

This page runs against [moto](https://docs.getmoto.org/), which fakes AWS in memory, so every output below is real and needs no AWS account. In your own code, skip the moto cell and create the client from your own boto3 session.

In [1]:
# Demo only: fake AWS in memory. Skip this cell in your own code.
import moto

mock_aws = moto.mock_aws()
mock_aws.start()

In [2]:
import boto3

boto_ses = boto3.Session(region_name="us-east-1")
lambda_client = boto_ses.client("lambda")

## Publish a Layer to Play With

`simple_aws_lambda` reads layers; publishing is a plain boto3 call. Publish three versions of one layer:

In [3]:
layer_name = "my_layer"

for i in range(1, 4):
    lambda_client.publish_layer_version(
        LayerName=layer_name,
        Content={"ZipFile": f"version {i}".encode()},
        CompatibleRuntimes=["python3.12", "python3.13"],
        CompatibleArchitectures=["x86_64", "arm64"],
        Description=f"version {i}",
    )

## Import the Public API

Everything a user needs is in `simple_aws_lambda.api`:

In [4]:
import simple_aws_lambda.api as sal

## List Layers

In [5]:
layers = sal.list_layers(lambda_client=lambda_client).all()
layers

[Layer(_data={'LayerName': 'my_layer', 'LayerArn': 'arn:aws:lambda:us-east-1:123456789012:layer:my_layer', 'LatestMatchingVersion': {'LayerVersionArn': 'arn:aws:lambda:us-east-1:123456789012:layer:my_layer:3', 'Version': 3, 'Description': 'version 3', 'CreatedDate': '2026-10-20T17:56:04.145+0000', 'CompatibleArchitectures': ['x86_64', 'arm64'], 'CompatibleRuntimes': ['python3.12', 'python3.13'], 'LicenseInfo': ''}})]

Each layer also carries its latest matching version:

In [6]:
layers[0].latest_matching_version.version

3

## List Layer Versions

In [7]:
layer_versions = sal.list_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
).all()
for layer_version in layer_versions:
    print(layer_version.version, layer_version.layer_version_arn)

3 arn:aws:lambda:us-east-1:123456789012:layer:my_layer:3
2 arn:aws:lambda:us-east-1:123456789012:layer:my_layer:2
1 arn:aws:lambda:us-east-1:123456789012:layer:my_layer:1


`compatible_runtime` and `compatible_architecture` filter on the AWS side, and `max_items` caps how many versions are read:

In [8]:
layer_versions = sal.list_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
    compatible_runtime="python3.12",
    max_items=2,
).all()
[layer_version.version for layer_version in layer_versions]

[3, 2]

## Get One Layer Version

In [9]:
layer_version = sal.get_layer_version(
    lambda_client=lambda_client,
    layer_name=layer_name,
    version_number=2,
)
layer_version.core_data

{'layer_version_arn': 'arn:aws:lambda:us-east-1:123456789012:layer:my_layer:2',
 'layer_arn': 'arn:aws:lambda:us-east-1:123456789012:layer:my_layer',
 'version': 2,
 'description': 'version 2',
 'created_date': '2026-10-20T17:56:04.144+0000',
 'compatible_runtimes': ['python3.12', 'python3.13'],
 'compatible_architectures': ['x86_64', 'arm64']}

A version that does not exist returns `None` instead of raising:

In [10]:
layer_version = sal.get_layer_version(
    lambda_client=lambda_client,
    layer_name=layer_name,
    version_number=999,
)
print(f"{layer_version = }")

layer_version = None


## Get the Latest Layer Version

In [11]:
latest = sal.get_latest_layer_version(
    lambda_client=lambda_client,
    layer_name=layer_name,
)
latest.version

3

In [12]:
latest = sal.get_latest_layer_version(
    lambda_client=lambda_client,
    layer_name="layer_never_published",
)
print(f"{latest = }")

latest = None


Stop moto when the demo is done:

In [13]:
mock_aws.stop()